# FairHire AI — Data Analysis Notebook

This notebook connects to the FairHire AI SQLite database and lets you:
- Explore evaluation data
- Analyze scoring distributions
- Review bias test results
- Audit user activity
- Track AI usage costs
- Visualize compliance metrics

## Setup
Run this cell first to connect to the database.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import json
from datetime import datetime

# Connect to the FairHire AI database
# Change the path if your database is in a different location
DB_PATH = '../data/fairhire.db'  # if notebook is in fairhire-ai/notebooks/
# DB_PATH = 'data/fairhire.db'  # if notebook is in fairhire-ai/

conn = sqlite3.connect(DB_PATH)
print('Connected to FairHire AI database')

# Show all tables
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f'\nTables found: {len(tables)}')
for t in tables['name']:
    count = pd.read_sql_query(f'SELECT COUNT(*) as count FROM {t}', conn).iloc[0]['count']
    print(f'  {t}: {count} rows')

## 1. Evaluation Overview

In [ ]:
# Load all evaluations with job and candidate info
evals = pd.read_sql_query("""
    SELECT e.*, c.original_filename, j.title as job_title
    FROM evaluations e
    LEFT JOIN candidates c ON e.candidate_id = c.id
    LEFT JOIN job_descriptions j ON e.job_id = j.id
    ORDER BY e.created_at DESC
""", conn)

print(f'Total evaluations: {len(evals)}')
print(f'\nQualification breakdown:')
print(evals['qualification'].value_counts())
print(f'\nAverage scores:')
print(f'  Skills:     {evals["skills_match_score"].mean():.1f}')
print(f'  Experience: {evals["experience_score"].mean():.1f}')
print(f'  Education:  {evals["education_score"].mean():.1f}')
print(f'  Overall:    {evals["overall_score"].mean():.1f}')
evals.head(10)

## 2. Score Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('FairHire AI — Score Distributions', fontsize=14, fontweight='bold')

scores = [
    ('skills_match_score', 'Skills Match', '#1a9e8f'),
    ('experience_score', 'Experience', '#2bb673'),
    ('education_score', 'Education', '#163352'),
    ('overall_score', 'Overall', '#0d5e6b'),
]

for ax, (col, label, color) in zip(axes.flat, scores):
    data = evals[col].dropna()
    if len(data) > 0:
        ax.hist(data, bins=10, range=(0, 10), color=color, alpha=0.8, edgecolor='white')
        ax.axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.1f}')
        ax.set_title(label)
        ax.set_xlabel('Score')
        ax.set_ylabel('Count')
        ax.legend()
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
        ax.set_title(label)

plt.tight_layout()
plt.show()

## 3. Qualification by Job

In [ ]:
if len(evals) > 0:
    qual_by_job = evals.groupby(['job_title', 'qualification']).size().unstack(fill_value=0)
    colors = {'Meets requirements': '#2bb673', 'Partially meets requirements': '#e5a832', 'Does not meet requirements': '#e05858'}
    col_colors = [colors.get(c, '#999') for c in qual_by_job.columns]
    qual_by_job.plot(kind='bar', stacked=True, figsize=(10, 5), color=col_colors)
    plt.title('Qualification Results by Job Position')
    plt.xlabel('Job Title')
    plt.ylabel('Count')
    plt.legend(title='Qualification', bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()
else:
    print('No evaluations to plot')

## 4. Bias Test Results

In [ ]:
bias = pd.read_sql_query('SELECT * FROM bias_test_results ORDER BY created_at DESC', conn)

if len(bias) > 0:
    print(f'Total bias tests: {len(bias)}\n')
    for _, row in bias.iterrows():
        status = 'PASS' if row['passed_80_rule'] else 'FAIL'
        print(f"{row['test_name']}")
        print(f"  {row['group_a_label']}: {row['group_a_pass_rate']*100:.0f}% pass rate")
        print(f"  {row['group_b_label']}: {row['group_b_pass_rate']*100:.0f}% pass rate")
        print(f"  DI Ratio: {row['disparate_impact_ratio']:.3f} — {status}")
        print(f"  Date: {row['created_at']}")
        print()
    
    # Plot DI ratios
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#2bb673' if r >= 0.8 else '#e05858' for r in bias['disparate_impact_ratio']]
    bars = ax.barh(bias['test_name'] + ' (' + bias['created_at'].str[:10] + ')', bias['disparate_impact_ratio'], color=colors)
    ax.axvline(0.8, color='red', linestyle='--', linewidth=1.5, label='80% Threshold')
    ax.set_xlabel('Disparate Impact Ratio')
    ax.set_title('Bias Test Results — Disparate Impact Ratios')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No bias tests found. Run: npm run test:bias')

## 5. User Activity Audit

In [ ]:
activity = pd.read_sql_query('SELECT * FROM user_activity_logs ORDER BY created_at DESC LIMIT 100', conn)

if len(activity) > 0:
    print(f'Recent activity entries: {len(activity)}\n')
    print('Actions by type:')
    print(activity['action'].value_counts())
    print(f'\nActions by user:')
    print(activity['user_email'].value_counts())
    print()
    activity[['user_email', 'role', 'action', 'created_at']].head(20)
else:
    print('No user activity logs found')

## 6. AI Usage & Cost Tracking

In [ ]:
usage = pd.read_sql_query('SELECT * FROM usage_tracking ORDER BY month DESC', conn)

if len(usage) > 0:
    print('AI Usage by Month:\n')
    for _, row in usage.iterrows():
        print(f"{row['month']}: {row['request_count']} requests, "
              f"{row['total_tokens']:,} tokens, ${row['estimated_cost']:.4f}")
    
    if len(usage) > 1:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.bar(usage['month'], usage['total_tokens'], color='#1a9e8f')
        ax1.set_title('Tokens Used by Month')
        ax1.set_ylabel('Total Tokens')
        ax2.bar(usage['month'], usage['estimated_cost'], color='#2bb673')
        ax2.set_title('Cost by Month ($)')
        ax2.set_ylabel('USD')
        ax2.axhline(5.0, color='red', linestyle='--', label='$5 Cap')
        ax2.legend()
        plt.tight_layout()
        plt.show()
else:
    print('No usage data yet (only tracked when Claude API is active, not in mock mode)')

## 7. Recruiter Overrides Analysis

In [ ]:
overrides = pd.read_sql_query("""
    SELECT ro.*, e.qualification as ai_qualification, e.overall_score,
           c.original_filename, j.title as job_title
    FROM recruiter_overrides ro
    LEFT JOIN evaluations e ON ro.evaluation_id = e.id
    LEFT JOIN candidates c ON e.candidate_id = c.id
    LEFT JOIN job_descriptions j ON e.job_id = j.id
    ORDER BY ro.created_at DESC
""", conn)

if len(overrides) > 0:
    print(f'Total overrides: {len(overrides)}\n')
    print('Override patterns:')
    for _, row in overrides.iterrows():
        print(f"  {row['original_filename']}: {row['original_qualification']} → {row['new_qualification']}")
        if row['notes']:
            print(f"    Notes: {row['notes']}")
else:
    print('No recruiter overrides recorded yet')

## 8. HR Certifications

In [ ]:
certs = pd.read_sql_query("""
    SELECT hc.*, j.title as job_title
    FROM hr_certifications hc
    LEFT JOIN job_descriptions j ON hc.job_id = j.id
    ORDER BY hc.certified_at DESC
""", conn)

if len(certs) > 0:
    print(f'Total certifications: {len(certs)}\n')
    for _, row in certs.iterrows():
        print(f"Job: {row['job_title']}")
        print(f"  Certified by: {row['certified_by_name']} ({row['certified_by_email']})")
        print(f"  Candidates reviewed: {row['candidates_reviewed']}")
        print(f"  Date: {row['certified_at']}")
        if row['notes']:
            print(f"  Notes: {row['notes']}")
        print()
else:
    print('No HR certifications recorded yet')

## 9. Dataset Browser

In [ ]:
dataset = pd.read_sql_query('SELECT * FROM dataset_records', conn)

if len(dataset) > 0:
    print(f'Dataset records: {len(dataset)}\n')
    print('Categories:')
    print(dataset['category'].value_counts())
    
    fig, ax = plt.subplots(figsize=(8, 4))
    dataset['category'].value_counts().plot(kind='bar', color='#1a9e8f', ax=ax)
    ax.set_title('Resume Dataset by Category')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No dataset records. Run: npm run seed')

## 10. Custom SQL Query
Write your own SQL query below.

In [ ]:
# Write any SQL query here
query = """
SELECT qualification, 
       COUNT(*) as count,
       AVG(overall_score) as avg_score,
       MIN(overall_score) as min_score,
       MAX(overall_score) as max_score
FROM evaluations
GROUP BY qualification
ORDER BY avg_score DESC
"""

result = pd.read_sql_query(query, conn)
result

In [ ]:
# Always close the connection when done
conn.close()
print('Database connection closed')